# Part C — Churn Classification

Target: Churn

Compare classification models, tune Random Forest, and identify important churn predictors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve


In [ ]:
# Load dataset
df = pd.read_csv('../data/customer_360_ml_workshop.csv')
df.head()

In [ ]:
# Prepare target
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)

X = df.drop(columns=['Churn', 'ChurnFlag', 'CustomerID'])
y = df['ChurnFlag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True),
    'Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    results.append([name,
                    accuracy_score(y_test,pred),
                    precision_score(y_test,pred),
                    recall_score(y_test,pred),
                    f1_score(y_test,pred),
                    roc_auc_score(y_test,prob)])
    trained_models[name] = pipe

results_df = pd.DataFrame(results, columns=['Model','Accuracy','Precision','Recall','F1','ROC-AUC'])
results_df.sort_values('ROC-AUC', ascending=False)

In [ ]:
# ROC Curve
plt.figure(figsize=(9,6))
for name, model in trained_models.items():
    prob = model.predict_proba(X_test)[:,1]
    fpr,tpr,_ = roc_curve(y_test,prob)
    plt.plot(fpr,tpr,label=name)

plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
# Confusion Matrix
best_model_name = results_df.sort_values('ROC-AUC', ascending=False).iloc[0]['Model']
best_model = trained_models[best_model_name]

cm = confusion_matrix(y_test, best_model.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Random Forest GridSearchCV
rf_pipe = Pipeline([('prep', preprocessor),
                    ('model', RandomForestClassifier(random_state=42))])

params = {
    'model__n_estimators':[100,200],
    'model__max_depth':[5,10,None],
    'model__min_samples_split':[2,5]
}

grid = GridSearchCV(rf_pipe, params, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train,y_train)

print(grid.best_params_)
print(grid.best_score_)

In [ ]:
# Feature Importance
rf = grid.best_estimator_.named_steps['model']
prep = grid.best_estimator_.named_steps['prep']

features = prep.get_feature_names_out()

importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

importance.head(15)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=importance.head(15), x='Importance', y='Feature')
plt.title('Top Churn Predictors')
plt.show()